# Advertising Optimal Read Chunk Shape

This notebook demonstrates the `read_chunk_shape` field proposed in
[zarr-conventions/multiscales#9](https://github.com/zarr-conventions/multiscales/issues/9).
When present in a multiscales layout entry, `read_chunk_shape` tells clients
the smallest independently-readable chunk for each pyramid level — without
requiring them to fetch each level's `zarr.json` and parse its codec chain.

We use [topozarr](https://github.com/zarr-conventions/topozarr) to build a
multiscale pyramid from an xarray Dataset, then inspect the resulting metadata
with [geozarr-toolkit](https://github.com/zarr-conventions/geozarr-toolkit).

**Prerequisites:** [Composition](composition.ipynb) | [COG to Zarr](cog-to-zarr.ipynb)

## Step 1: Create a sample dataset

We create a small synthetic dataset with two variables at different dtypes,
assign a CRS via xproj, and build a 3-level pyramid.

In [1]:
import json

import numpy as np
import xarray as xr
import xproj  # noqa: F401

np.random.seed(42)
nx, ny = 512, 512

ds = xr.Dataset(
    {
        "elevation": (("y", "x"), np.random.rand(ny, nx).astype("float32")),
        "temperature": (
            ("y", "x"),
            (np.random.rand(ny, nx) * 30 + 270).astype("float32"),
        ),
    },
    coords={
        "x": np.linspace(500_000, 500_000 + nx * 100, nx),
        "y": np.linspace(4_600_000, 4_600_000 - ny * 100, ny),
    },
)
ds = ds.proj.assign_crs(spatial_ref="EPSG:32633")
print(ds)

<xarray.Dataset> Size: 2MB
Dimensions:      (y: 512, x: 512)
Coordinates:
    spatial_ref  int64 8B 0
  * x            (x) float64 4kB 5e+05 5.001e+05 ... 5.511e+05 5.512e+05
  * y            (y) float64 4kB 4.6e+06 4.6e+06 ... 4.549e+06 4.549e+06
Data variables:
    elevation    (y, x) float32 1MB 0.5488 0.7152 0.6028 ... 0.07959 0.5247
    temperature  (y, x) float32 1MB 274.3 284.0 290.1 ... 277.3 289.9 278.6


## Step 2: Build the pyramid with topozarr

topozarr's `create_pyramid` produces a `Pyramid` object containing an xarray
DataTree and encoding metadata. It automatically computes chunk and shard sizes
optimized for web access, and writes `read_chunk_shape` into each level of the
multiscales layout.

In [2]:
from topozarr import create_pyramid

pyramid = create_pyramid(ds, levels=3)

print(f"Pyramid levels: {len(pyramid.encoding)}")
print("\nEncoding per level:")
for path, enc in sorted(pyramid.encoding.items()):
    var_name = next(iter(enc))
    chunks = enc[var_name]["chunks"]
    shards = enc[var_name].get("shards", chunks)
    print(f"  {path}: {var_name} chunks={chunks}, shards={shards}")

Pyramid levels: 3

Encoding per level:
  /0: elevation chunks=(256, 256), shards=(512, 512)
  /1: elevation chunks=(256, 256), shards=(256, 256)
  /2: elevation chunks=(128, 128), shards=(128, 128)


## Step 3: Inspect the multiscales metadata

The `read_chunk_shape` field appears on each layout entry. For sharded arrays,
this is the **inner chunk** shape (the smallest independently-readable unit).
For non-sharded arrays, it equals the array's chunk shape.

A client like OpenLayers can read this value directly from the group metadata
instead of fetching each level's `zarr.json` and parsing its codec chain.

In [3]:
multiscales = pyramid.datatree.attrs["multiscales"]
print(json.dumps(multiscales, indent=2))

{
  "layout": [
    {
      "asset": "0",
      "transform": {
        "scale": [1.0, 1.0],
        "translation": [0.0, 0.0]
      },
      "read_chunk_shape": [256, 256],
      "spatial:transform": [100.0, 0.0, 499950.0, 0.0, -100.0, 4600050.0],
      "spatial:shape": [512, 512]
    },
    {
      "asset": "1",
      "transform": {
        "scale": [2.0, 2.0],
        "translation": [0.5, 0.5]
      },
      "read_chunk_shape": [256, 256],
      "derived_from": "0",
      "resampling_method": "mean",
      "spatial:transform": [200.0, 0.0, 499900.0, 0.0, -200.0, 4600100.0],
      "spatial:shape": [256, 256]
    },
    {
      "asset": "2",
      "transform": {
        "scale": [2.0, 2.0],
        "translation": [0.5, 0.5]
      },
      "read_chunk_shape": [128, 128],
      "derived_from": "1",
      "resampling_method": "mean",
      "spatial:transform": [400.0, 0.0, 499800.0, 0.0, -400.0, 4600200.0],
      "spatial:shape": [128, 128]
    }
  ],
  "resampling_method": "mean"
}


## Step 4: Write to Zarr and validate with geozarr-toolkit

We write the pyramid to a local Zarr V3 store and use geozarr-toolkit to
validate and inspect the result. The `read_chunk_shape` field is preserved
in the stored metadata and recognized by the toolkit.

In [4]:
output_path = "data/read_chunk_shape_demo.zarr"

pyramid.datatree.to_zarr(
    output_path, mode="w", encoding=pyramid.encoding, zarr_format=3
)
print(f"Wrote Zarr V3 store to {output_path}")

Wrote Zarr V3 store to data/read_chunk_shape_demo.zarr


In [5]:
import zarr
from geozarr_toolkit import detect_conventions, validate_group

root = zarr.open_group(output_path, mode="r")

detected = detect_conventions(dict(root.attrs))
print(f"Detected conventions: {detected}")

results = validate_group(root)
for conv, errors in results.items():
    status = "PASS" if not errors else "FAIL"
    print(f"  [{status}] {conv}")
    for err in errors:
        print(f"         {err}")

Detected conventions: ['spatial', 'proj', 'multiscales']
  [PASS] spatial
  [PASS] proj
  [PASS] multiscales
  [PASS] zarr_conventions


## Step 5: Use read_chunk_shape with geozarr-toolkit

The `create_multiscales_layout` helper in geozarr-toolkit also supports
`read_chunk_shape`. This is useful when building multiscales metadata
manually (e.g., when converting from COG or other formats).

In [6]:
from geozarr_toolkit import create_multiscales_layout

layout = create_multiscales_layout(
    [
        {
            "asset": "0",
            "transform": {"scale": [1.0, 1.0]},
            "read_chunk_shape": [256, 256],
        },
        {
            "asset": "1",
            "derived_from": "0",
            "transform": {"scale": [2.0, 2.0]},
            "read_chunk_shape": [256, 256],
        },
        {
            "asset": "2",
            "derived_from": "0",
            "transform": {"scale": [4.0, 4.0]},
            "read_chunk_shape": [128, 128],
        },
    ]
)
print(json.dumps(layout, indent=2))

{
  "multiscales": {
    "layout": [
      {
        "asset": "0",
        "transform": {
          "scale": [1.0, 1.0]
        },
        "read_chunk_shape": [256, 256]
      },
      {
        "asset": "1",
        "transform": {
          "scale": [2.0, 2.0]
        },
        "read_chunk_shape": [256, 256],
        "derived_from": "0"
      },
      {
        "asset": "2",
        "transform": {
          "scale": [4.0, 4.0]
        },
        "read_chunk_shape": [128, 128],
        "derived_from": "0"
      }
    ]
  }
}


## Summary

The `read_chunk_shape` field in the multiscales layout tells clients the optimal
chunk size for reading each pyramid level:

- **For sharded arrays:** the inner (sub-)chunk shape — the unit a client fetches
  via byte-range request after reading the shard index
- **For non-sharded arrays:** the array's chunk shape

This avoids the need to fetch each level's `zarr.json` and parse its codec chain,
which is especially valuable for tiled map clients like OpenLayers that need this
information before issuing their first data request.

See the [proposal](https://github.com/zarr-conventions/multiscales/issues/9) for
full details and open questions.